# 12 · What each model leans on

Each fitted model is explained on the held-out women, with their survey weights:

- **Logistic regression** is additive: its log-odds is a sum over predictors, so a woman's
  contribution from a predictor is its coefficients times her values, measured from the
  weighted average woman. The split is exact.
- **Random forest and XGBoost** are explained by **TreeSHAP**, the exact Shapley attribution for
  a tree ensemble (probability units for the forest, log-odds for XGBoost).

A predictor's importance is its weighted mean absolute contribution; its **share** is that over
the model's total, which makes the models comparable even though their units differ. The
agreement table lists, per set and rung, each predictor's share under every model and how many
models put it in their top five; the concordance table is the rank correlation between models.

**Writes:** `outputs/explain/importance.csv`, `effects.json`, `agreement.csv`, `concordance.csv`

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

In [ ]:
# notebook 11 already computed them for the model cards
if not (P.outputs / "explain" / "importance.csv").exists():
    repro.run_script("run_explainability.py")

In [ ]:
agree = pd.read_csv(P.outputs / "explain" / "agreement.csv")
models = [m for m in ["Logistic regression", "Random Forest", "XGBoost", "Explainable Boosting"] if m in agree]
for rung in lib.RUNGS:
    print(rung, "· full pool: the ten predictors the models agree on most")
    print(agree[(agree.set == "pool") & (agree.rung == rung)].head(10)[["predictor", *models, "mean_rank", "top5_models"]].round(3).to_string(index=False), "\n")
print(pd.read_csv(P.outputs / "explain" / "concordance.csv").query("set == 'pool'").round(2).to_string(index=False))

In [ ]:
# The direction: each model's average contribution from the wealth quintile, by quintile, at ANC8+
# (log-odds for logistic regression and XGBoost, probability for the random forest)
eff = json.load(open(P.outputs / "explain" / "effects.json"))
order = ["Poorest", "Poorer", "Middle", "Richer", "Richest"]
table = pd.DataFrame({m: {r["level"]: r["effect"] for r in eff["pool"]["anc8plus"][m]["wealth_quintile"]} for m in models})
print(table.reindex(order).round(3).to_string())
repro.compare_table(pd.read_csv(P.outputs / "explain" / "importance.csv"), "explain_importance.csv",
                    keys=["set", "rung", "model", "predictor"], value="share", label="every predictor's share, per model")
repro.report()